In [1]:
import sys

!which {sys.executable}

/ut3/jerome/miniconda3/envs/wakai/bin/python


In [2]:
import sys
sys.path.insert(0, '../code')

# NExSci catalog

In [3]:
from catalog import get_nexsci_data

table = 'pscomppars'
nexsci = get_nexsci_data(table, clobber=False)
nexsci.tail()

Error.  nthreads cannot be larger than environment variable "NUMEXPR_MAX_THREADS" (64)2026-04-26 18:18:18.223 | INFO     | catalog:get_nexsci_data:1596 - Column definitions: 
2026-04-26 18:18:19.056 | INFO     | catalog:get_nexsci_data:1608 - Loaded: /ut2/jerome/github/research/project/wakai/data/nexsci_pscomppars.csv


,objectid,pl_name,pl_letter,hostid,hostname,hd_name,hip_name,tic_id,disc_pubdate,disc_year,...,pl_angseperr1,pl_angseperr2,pl_angseplim,pl_angsepformat,pl_angsepstr,pl_angsepsymerr,pl_angsep_reflink,pl_ndispec,sky_coord.ra,sky_coord.dec
4632,3.12663,TOI-178 d,d,2.104480,TOI-178,NaN,NaN,TIC 251848941,2021-05,2021,...,NaN,NaN,0.0,NaN,0.944,NaN,<a refstr=CALCULATED_VALUE href=/docs/pscp_cal...,0,7.302011,-30.454116
4633,3.18824,TOI-6109 c,c,2.578503,TOI-6109,NaN,NaN,TIC 384984325,2025-09,2025,...,NaN,NaN,0.0,NaN,0.518,NaN,<a refstr=CALCULATED_VALUE href=/docs/pscp_cal...,0,50.131921,42.606956
4634,3.12748,TOI-2076 b,b,2.572939,TOI-2076,NaN,NaN,TIC 27491137,2021-08,2021,...,NaN,NaN,0.0,NaN,2.100,NaN,<a refstr=CALCULATED_VALUE href=/docs/pscp_cal...,0,217.392016,39.790399
4635,3.17077,TOI-4364 b,b,2.576876,TOI-4364,NaN,NaN,TIC 4070275,2025-03,2025,...,NaN,NaN,0.0,NaN,1.080,NaN,<a refstr=CALCULATED_VALUE href=/docs/pscp_cal...,0,80.066927,-4.239248
4636,3.19428,TOI-198 b,b,2.539370,TOI-198,NaN,HIP 738,TIC 12421862,2023-03,2023,...,NaN,NaN,0.0,NaN,2.840,NaN,<a refstr=CALCULATED_VALUE href=/docs/pscp_cal...,0,2.271509,-27.121745


In [4]:
nexsci.shape

(4637, 685)

In [5]:
nexsci.hostname.unique().shape

(3553,)

# cross-match nexsci with Gaia DR3 using mast.Catalogs

In [18]:
len(nexsci.tic_id.unique())

3551

In [33]:
tic_ids = nexsci.tic_id.apply(lambda x: str(x).split()[1] if len(str(x).split())>1 else x).dropna().astype(int).unique()
len(tic_ids)

3550

In [31]:
from astroquery.mast import Catalogs
import numpy as np

tic_data = Catalogs.query_criteria(catalog="TIC", ID=tic_ids)

In [32]:
# Extract unique Gaia IDs and remove masked/null values
gaia_ids = [int(gid) for gid in tic_data['GAIA'] if gid is not np.ma.masked]
len(gaia_ids)

3546

In [34]:
tic_data.to_pandas().to_csv('../data/nexsci_tic_data.csv', index=False)

In [48]:
tic_data['tic_id'] = tic_data.to_pandas().ID.apply(lambda x: f'TIC {x}')

In [53]:
import pandas as pd

df_merge = pd.merge(nexsci, tic_data.to_pandas(), on='tic_id')
df_merge['GAIA']

0       2086441721667415424
1       2052381737659167488
2       2100418850915010432
3       2078049767888231936
4       2136207095405014912
               ...         
4629    2318295979126499200
4630     241035591876195328
4631    1490845584382687232
4632    3210444215030339584
4633    2333676738049780352
Name: GAIA, Length: 4634, dtype: object

In [56]:
df_merge.to_csv('../data/nexsci_with_Gaiaid.csv', index=False)

# cross-match nexsci with Gaia DR3 using `astroquery.xmatch`

In [6]:
import astropy.units as u
from astroquery.xmatch import XMatch
import pandas as pd

gaia_table = XMatch.query(
    cat1=open(f'../data/nexsci_{table}.csv'),
    cat2="vizier:I/355/gaiadr3",  # The Vizier identifier for Gaia DR3
    max_distance=1 * u.arcsec,    # Set the maximum matching radius
    colRA1="ra",  # Column name for Right Ascension in TIC results
    colDec1="dec" # Column name for Declination in TIC results
)
gaia_df = gaia_table.to_pandas()

# results to 6237 matches
matches = gaia_df.copy()

# Apply magnitude filtering criteria
# For main-sequence stars, the difference between TESS T and Gaia G is around 0–0.5 magnitudes
# Redder stars might have T - G values closer to 1–2 magnitudes.
# For giants or highly reddened stars, the difference could be larger.
if False:
    # results to 5971 matches
    matches = []
    for i, row in gaia_df.iterrows():
        teff=row['st_teff']
        if teff<3500:
            magnitude_threshold = 2
        elif (teff>3500) and (teff<6000):
            magnitude_threshold = 1
        else:
            magnitude_threshold = 1.5
        if abs(row['Gmag'] - row['TESS Mag']) < magnitude_threshold:
            matches.append(row)
    matches = pd.concat(matches, axis=1).T
if False:
    # results to 5700 matches
    magnitude_threshold = 1
    matches = gaia_df[abs(gaia_df['Gmag'] - gaia_df['sy_tmag']) < magnitude_threshold]
if False:
    # results to 4878 matches
    teff_threshold = 1000
    matches = gaia_df[abs(gaia_df['Teff'] - gaia_df['st_teff']) < teff_threshold]
matches.head()

Could not import regions, which is required for some of the functionalities of this module.


,angDist,objectid,pl_name,pl_letter,hostid,hostname,hd_name,hip_name,tic_id,disc_pubdate,...,B_AG,E(BP-RP),b_E(BP-RP),B_E(BP-RP),Lib,RAJ2000,DEJ2000,e_RAJ2000,e_DEJ2000,RADEcorJ2000
0,0.000790,3.23620,Kepler-1812 b,b,2.454097,Kepler-1812,NaN,NaN,TIC 170346516,2022-02,...,0.3254,0.1656,0.1590,0.1751,MARCS,298.084515,40.283253,0.391880,0.407567,-0.0730
1,0.004019,3.25790,Kepler-1822 b,b,2.476824,Kepler-1822,NaN,NaN,TIC 169177842,2022-02,...,0.1343,0.0468,0.0283,0.0714,MARCS,297.285489,41.305606,0.476402,0.491835,-0.1586
2,0.026341,3.91950,K2-347 b,b,2.198018,K2-347,NaN,NaN,TIC 14603874,2016-10,...,0.2006,0.0961,0.0906,0.1058,MARCS,126.372926,23.297377,0.420547,0.296408,-0.3926
3,0.005877,3.10327,K2-343 c,c,2.193496,K2-343,NaN,NaN,TIC 197246627,2018-07,...,0.4557,0.2620,0.2602,0.2636,MARCS,134.202226,21.502673,0.579617,0.382229,-0.2577
4,0.009381,3.91440,K2-332 b,b,2.164596,K2-332,NaN,NaN,TIC 330621494,2016-10,...,0.5747,0.3649,0.3576,0.3783,MARCS,135.883336,14.250545,0.982793,0.762718,-0.2331


In [7]:
matches.shape

(4313, 822)

In [8]:
matches.hostname.unique().shape

(3235,)

In [9]:
matches['GaiaDR3'] = matches['DR3Name'].apply(lambda x: int(x.split()[-1]))
matches.to_csv('../data/nexsci_GaiaDR3_xmatch.csv', index=False)

In [10]:
idx = nexsci.hostname.isin(matches.hostname.unique())
nexsci[~idx].shape

(2, 684)

In [11]:
nexsci[~idx].to_csv('../data/nexsci_GaiaDR3_xmatch_missing.csv', index=False)